# Cue-invariant 8B fresh-confirmation extraction

Upload this notebook to the Google Colab website, choose an A100 GPU under **Runtime > Change runtime type**, and run the cells from top to bottom. It mounts Google Drive directly and reads the six frozen inputs from `MyDrive/lang-ea/`. It extracts the unchanged fitting items, the spent development controls used to construct the nuisance subspace, and the second fresh controls in one unquantized model environment. Only mean layer 6 is stored. No probe is fitted and naturalistic transfer is never loaded.

Before running, place the six files listed in `CUE_INVARIANT_CONFIRMATION_HANDOFF.md` directly in `MyDrive/lang-ea/`. The two model cells are independent and outputs persist in `MyDrive/lang-ea/cue_invariant_confirmatory_8b/`. If Colab disconnects after one model, reconnect to an A100 and run from the top; verified completed output is skipped.

In [ ]:
%pip install -q transformers accelerate sentencepiece

In [ ]:
from pathlib import Path
import json, subprocess, sys, torch

BATCH_SIZE = 8
SKIP_COMPLETE = True
SWALLOW = "tokyotech-llm/Llama-3.1-Swallow-8B-Instruct-v0.3"
META = "unsloth/Meta-Llama-3.1-8B-Instruct"
MODELS = [SWALLOW, META]
EXPECTED_ITEMS = "97796dcf3f54918cb7b55c009f6458832fcf75d880df50252ffdf8415c6bc84e"
EXPECTED_DEVELOPMENT = "fbd320168c38e14321e50cab1872144a3d32e171d4a56afca63dcb6f782131b9"
EXPECTED_FRESH = "0f93681d626c2e4d6d7c91acf9c8301f42f1203f45136d32441a2ef68de654fb"

from google.colab import drive
drive.mount("/content/drive", force_remount=False)
DRIVE_DIR = Path("/content/drive/MyDrive/lang-ea")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
EXTRACTOR = DRIVE_DIR / "extract_cue_invariant_confirmatory_8b.py"
HELPER = DRIVE_DIR / "extract_confirmatory_8b.py"
ITEMS = DRIVE_DIR / "probe_items.jsonl"
DEVELOPMENT = DRIVE_DIR / "format_matched_controls.jsonl"
DEVELOPMENT_RESULT = DRIVE_DIR / "cue_invariant_development.json"
FRESH = DRIVE_DIR / "cue_invariant_confirmatory_controls.jsonl"
required = [EXTRACTOR, HELPER, ITEMS, DEVELOPMENT, DEVELOPMENT_RESULT, FRESH]
missing = [path.name for path in required if not path.is_file()]
if missing: raise FileNotFoundError(f"Missing from {DRIVE_DIR}: {missing}")
WORKDIR = EXTRACTOR.parent
OUTPUT = DRIVE_DIR / "cue_invariant_confirmatory_8b"
assert torch.cuda.is_available(), "Select the A100 GPU runtime."
gpu = torch.cuda.get_device_properties(0)
assert "A100" in gpu.name.upper(), f"A100 required; found {gpu.name}"
print(f"GPU: {gpu.name}; output: {OUTPUT}")
print(f"items: {ITEMS}\ndevelopment: {DEVELOPMENT}\ndevelopment result: {DEVELOPMENT_RESULT}\nfresh: {FRESH}")

In [ ]:
# Hash-only preflight: no model or activations are loaded.
subprocess.run([sys.executable, str(EXTRACTOR), "--items", str(ITEMS), "--development-controls", str(DEVELOPMENT), "--development-result", str(DEVELOPMENT_RESULT), "--fresh-controls", str(FRESH), "--check-only"], cwd=WORKDIR, check=True)

In [ ]:
def model_dir(model): return OUTPUT / model.split("/")[-1]
def load_tensor(path):
    try: return torch.load(path, map_location="cpu", weights_only=True)
    except TypeError: return torch.load(path, map_location="cpu")
def verify(model):
    out = model_dir(model)
    cfg = json.loads((out / "config.json").read_text(encoding="utf-8"))
    assert cfg["model"] == model and cfg["load_in_4bit"] is False
    assert cfg["model_load_dtype"] == "bfloat16" and "A100" in cfg["accelerator"].upper()
    assert cfg["items_sha256"] == EXPECTED_ITEMS
    assert cfg["development_controls_sha256"] == EXPECTED_DEVELOPMENT
    assert cfg["fresh_controls_sha256"] == EXPECTED_FRESH
    assert cfg["gating_cell"] == {"position": "mean", "layer": 6}
    for directory, n in [(out,1200),(out/"development_controls",256),(out/"fresh_confirmation",256)]:
        assert len((directory/"meta.jsonl").read_text(encoding="utf-8").splitlines()) == n
        tensor = load_tensor(directory/"acts_layer6_mean.pt")
        assert tuple(tensor.shape) == (n,4096), tensor.shape
        del tensor
    print(f"VERIFIED: {model} -> {out}")
def run_model(model, force=False):
    if SKIP_COMPLETE and not force:
        try:
            verify(model); print("Complete output exists; skipping."); return
        except (AssertionError, FileNotFoundError, KeyError, json.JSONDecodeError): pass
    command = [sys.executable,str(EXTRACTOR),"--items",str(ITEMS),"--development-controls",str(DEVELOPMENT),"--development-result",str(DEVELOPMENT_RESULT),"--fresh-controls",str(FRESH),"--out",str(OUTPUT),"--models",model,"--batch-size",str(BATCH_SIZE)]
    OUTPUT.mkdir(parents=True, exist_ok=True)
    log_path = OUTPUT / f"{model.split('/')[-1]}.log"
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command,cwd=WORKDIR,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line); log.flush()
        returncode = process.wait()
    if returncode:
        raise RuntimeError(f"Extraction failed with exit code {returncode}. Full log: {log_path}")
    verify(model)

## 1. Swallow 8B

In [ ]:
run_model(SWALLOW)

## 2. Meta-Llama 8B

In [ ]:
run_model(META)

In [ ]:
for model in MODELS: verify(model)
print("BOTH EXTRACTIONS COMPLETE. Return both output directories; do not inspect fresh projections before running the frozen evaluator.")